# 2. 流程增强

这一节解决的问题是：**一次检索 + 一次生成不够时，系统应该如何多走几步。**

## 与第4章的边界

- 第4章强调：如何把 `query` 变得更适合检索。
- 本节强调：系统拿到中间结果后，如何继续决策下一步流程。

## 统一实验设置

与前一节保持一致：

- **数据**：`../3. 索引阶段/data/pumpkin_book.pdf`（南瓜书《机器学习公式详解》）
- **问答数据**：`../3. 索引阶段/data/train_dataset.json`（选取前 5 个问答对用于实验）
- **生成模型**：`glm-4-flash-250414`
- **向量模型**：本地 `BAAI/bge-small-zh-v1.5`
- **评估**：使用 LLM 作为裁判进行评估

## 环境准备

本节使用智谱 AI 的 `GLM-4-Flash` 做生成模型，使用本地 `BAAI/bge-small-zh-v1.5` 做 embedding。运行前请确保：

1. 安装依赖：`pip install langchain langchain-community langchain-chroma zhipuai python-dotenv pymupdf pandas modelscope sentence-transformers transformers torch`
2. 在项目根目录的 `.env` 文件中配置 `ZHIPUAI_API_KEY`
3. 首次运行会自动从 ModelScope 下载本地 embedding 模型到当前目录下的 `./models/`


In [ ]:
import os
import re
import json
import warnings
from dataclasses import dataclass, field
from typing import Any
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

import sys
sys.path.insert(0, ".")
from _common import (
    get_embeddings, get_cleaned_pdf_documents, open_or_build_chroma,
    llm_call as _raw_llm_call, build_rag_generation_prompt,
    trim_context_to_budget, load_qna_subset,
    simple_eval_2pt, run_shared_eval, build_compare_table,
    CHROMA_COLLECTION, CONTEXT_CHAR_BUDGET, PDF_PATH, QA_PATH,
)

warnings.filterwarnings("ignore")


def llm_call(prompt: str) -> str:
    """6.2 节默认每次成功调用后 sleep 1 秒，对抗多步流程的连续调用限流。"""
    return _raw_llm_call(prompt, sleep_after=1.0)


def load_chunks(chunk_size=256, chunk_overlap=20):
    docs = list(get_cleaned_pdf_documents())
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""], keep_separator=True,
    )
    return splitter.split_documents(docs)


def build_retriever(chunk_size=256, chunk_overlap=20, k=4):
    persist_dir = f"./chroma_db/baseline_{chunk_size}_{chunk_overlap}"
    chunks = load_chunks(chunk_size, chunk_overlap)
    ids = [f"b{i}" for i in range(len(chunks))]
    vs = open_or_build_chroma(persist_dir, chunks, ids)
    return vs.as_retriever(search_kwargs={"k": k})


retriever = build_retriever()
print("✅ 6.2 环境准备完成（公共底座来自 _common；本节默认 sleep_after=1.0）")


## 本节钩子：`*_pipeline(question) -> str`

6.1 节的钩子是 `*_context(question) -> str`——每个方法只决定"如何拼 context"，
拼完后由公共胶水接到 LLM。但流程增强的核心**就是**控制流：多轮检索、子问题、
路由、自反思。所以本节钩子升一层——**每个方法直接返回最终答案**：

```
*_pipeline(question) -> answer_str
```

由于 `*_pipeline` 也是 `q -> str` 的形状，公共底座的 `run_shared_eval` 不需要修改，
直接把 `*_pipeline` 当 `answer_fn` 喂进去即可。

为了让读者看到"流程的形状"，每个方法内部维护一个 `trace: list[Step]`，
`inspect_*` 顺次打印这些步骤——这是本节 inspect 的重点。


In [ ]:
@dataclass
class Step:
    """一次流程内的中间步骤。kind 例：retrieve / draft / route / reflect / finalize。"""
    kind: str
    payload: dict = field(default_factory=dict)


def print_trace(trace: list[Step]) -> None:
    for i, s in enumerate(trace):
        head = f"[{i+1}] {s.kind}"
        body = ", ".join(f"{k}={repr(v)[:60]}" for k, v in s.payload.items())
        print(f"  {head}  {body}")


## 本节评估口径：维度计分版 0~2 分

6.2 处理的是「多维度复杂问题」（如"对偶优势 + KKT + Slater"三个独立角度）。
朴素 0~2 分裁判会被语言流畅度干扰。本节定制 prompt：要求裁判先列出
**该题的必答要点**，再按要点覆盖比例映射到 0/1/2，输出仍是一行 0/1/2。
接口仍是 `simple_eval_2pt(answer, expected, question, prompt_template=...)`，
保证 `build_compare_table` 能与 6.1 / 6.3 拼接。


In [ ]:
DIMENSIONAL_EVAL_PROMPT = (
    "你是判卷人。先在心里把「参考答案」拆成 2~4 个【必答要点】"
    "（如：方法1、方法2、关键条件、关键定义等），不要输出这些要点。\n"
    "然后比对「模型答案」覆盖了几个要点：\n"
    "- 全部覆盖且无关键事实错误：2\n"
    "- 覆盖一半左右、或部分要点表述含糊但方向正确：1\n"
    "- 大部分要点缺失、或关键事实错误：0\n\n"
    "用户问题：{question}\n参考答案：{expected_answer}\n模型答案：{llm_answer}\n\n"
    "仅输出一行，只包含字符 0、1 或 2，不要任何其它文字。"
)


def eval_pipeline(answer_fn, qna_dict):
    """6.2 节的快捷入口：固定使用维度计分 prompt。"""
    return run_shared_eval(answer_fn, qna_dict, eval_prompt_template=DIMENSIONAL_EVAL_PROMPT)


In [ ]:
QA_INDICES = [0, 1, 3, 4, 7]  # 本节挑"多维度复杂"题；可按需调整
qna_dict = load_qna_subset(QA_PATH, QA_INDICES)
print(f"✅ 本节使用 {len(qna_dict)} 道多维度题（QA_INDICES={QA_INDICES}）")


## 六种方法的流程控制分类

| 方法 | 控制类型 | 关键决策点 |
|---|---|---|
| 迭代检索 | 补检索型 | "还缺什么？" -> 继续/停止 |
| 递归检索 | 拆任务型 | "该拆成哪些子问题？" |
| 查询路由 | 选路径型 | "走哪条检索链路？" |
| Corrective RAG | 控质量型 | "检索结果够好吗？" -> 过滤/补检 |
| Self-RAG | 自反思型 | "回答够好吗？" -> 继续检索/停止 |
| 自适应检索 | 选深度型 | "这个问题有多复杂？" -> 选择检索策略 |


## 迭代检索（Iterative Retrieval）

### 失败场景
单轮回答只覆盖了问题的一部分，遗漏关键证据。

### 循环流程
1. 初次检索并生成草答
2. 让 LLM 指出“还缺什么”
3. 用缺失点继续检索
4. 合并上下文再回答

### 边界
- 适合：信息可逐步补齐的问题
- 局限：轮次越多，延迟与成本越高

### 关键决策点
当前答案是否仍有关键缺口，若有则继续检索，否则停止。


In [ ]:
def iterative_pipeline(question: str, max_rounds: int = 2) -> str:
    trace: list[Step] = []
    context_blocks: list[str] = []
    missing_hint = ""
    merged_ctx = ""

    for i in range(max_rounds):
        effective_q = question if not missing_hint else f"{question}\n补充检索线索：{missing_hint}"
        docs = retriever.invoke(effective_q)
        context_blocks.extend(d.page_content for d in docs)
        merged_ctx = trim_context_to_budget("\n\n".join(context_blocks[-8:]), CONTEXT_CHAR_BUDGET)
        trace.append(Step("retrieve", {"round": i + 1, "hint": missing_hint, "n_hits": len(docs)}))

        draft_prompt = (
            f"问题：{question}\n上下文：\n{merged_ctx}\n"
            "请先给出当前答案，再用一句话指出仍缺失的关键信息。\n"
            "输出格式：\n当前答案：...\n缺失信息：..."
        )
        draft = llm_call(draft_prompt)
        trace.append(Step("draft", {"round": i + 1, "draft_head": draft[:80]}))

        if "缺失信息：无" in draft or "缺失信息: 无" in draft:
            trace.append(Step("finalize", {"reason": "complete_after_draft"}))
            iterative_pipeline.last_trace = trace  # 给 inspect 使用
            return draft

        missing_hint = draft.split("缺失信息")[-1].strip("：: \n")[:120]

    final = llm_call(build_rag_generation_prompt(question, merged_ctx))
    trace.append(Step("finalize", {"reason": "max_rounds_reached"}))
    iterative_pipeline.last_trace = trace
    return final


def inspect_iterative(question: str) -> None:
    print(f"❓ 问题: {question}\n")
    ans = iterative_pipeline(question)
    print(f"🧠 最终答案：\n{ans}\n")
    print("🔍 流程 trace：")
    print_trace(iterative_pipeline.last_trace)


inspect_iterative(list(qna_dict.keys())[0])


### 迭代检索结果分析

对比 baseline 和迭代检索的输出可以看到，迭代检索通过"先答后补"逐步完善了答案的覆盖度。第一轮给出初步回答并指出缺失（如未提到 KKT 条件），第二轮根据缺失线索补充检索，最终合并的上下文比单轮检索更全面。

不过迭代检索的子问题是**隐式产生**的——由 LLM 自行判断"还缺什么"。如果问题本身就可以显式拆解为独立的子问题（"对偶优势是什么"、"KKT 条件的作用"、"强对偶性条件"），直接拆比让模型自己猜更可控。这就是递归检索的思路。

## 递归检索（Recursive Retrieval）

### 失败场景
复杂问题包含多个子目标，单次检索无法覆盖全部维度。

### 分解流程
1. 主问题拆成子问题
2. 每个子问题独立检索与回答
3. 聚合子答案得到最终回答

### 子问题示例（针对题集中类似复合题）
- 为什么要对机器学习模型进行评估？
- 经验误差（训练误差）的定义是什么？
- 泛化误差的定义是什么？两者哪个更应该被最小化？

### 边界
- 适合：可显式分解的问题
- 局限：子问题质量决定上限

### 关键决策点
主问题是否可有效拆解为可检索、可验证的子问题。


In [ ]:
def recursive_pipeline(question: str, sub_questions: list | None = None) -> str:
    trace: list[Step] = []

    if sub_questions is None:
        decompose_prompt = (
            "任务：将以下复杂问题拆解为 2-3 个独立的、可检索的子问题。\n"
            f"主问题：{question}\n"
            "要求：仅输出子问题列表，每行一个，不要包含序号。"
        )
        raw = llm_call(decompose_prompt)
        sub_questions = [q.strip() for q in raw.split("\n") if q.strip()]
    if not sub_questions:
        sub_questions = [question]

    trace.append(Step("decompose", {
        "n_subs": len(sub_questions),
        "subs_head": [q[:40] for q in sub_questions],
    }))

    sub_answers = []
    for i, sq in enumerate(sub_questions):
        docs = retriever.invoke(sq)
        ctx = trim_context_to_budget("\n\n".join(d.page_content for d in docs), CONTEXT_CHAR_BUDGET)
        ans = llm_call(build_rag_generation_prompt(sq, ctx))
        sub_answers.append({"sub_question": sq, "answer": ans})
        trace.append(Step("sub_answer", {"idx": i + 1, "sub_q_head": sq[:60], "ans_head": ans[:60]}))

    merge_prompt = (
        f"主问题：{question}\n"
        f"子答案集：{sub_answers}\n"
        "请根据子答案集，整合为一个结构化的最终回答（包含：核心优势、关键条件、结论）。"
    )
    final = llm_call(merge_prompt)
    trace.append(Step("finalize", {"reason": "merged_from_subs"}))
    recursive_pipeline.last_trace = trace
    return final


def inspect_recursive(question: str) -> None:
    print(f"❓ 问题: {question}\n")
    ans = recursive_pipeline(question)
    print(f"🧠 最终答案：\n{ans}\n")
    print("🔍 流程 trace：")
    print_trace(recursive_pipeline.last_trace)


### 递归检索结果分析

递归检索的优势在于每个子问题都有独立的检索上下文，不会互相干扰。最终合成时 LLM 可以看到所有子答案，给出更结构化的回答。

但迭代和递归检索都在解决"多走几步"的问题，还没回答一个更基础的问题：**该走哪条路？** 当系统有多个索引或多个工具时，选错检索链路会导致无论走多少步都找不到对的信息。这就是查询路由要解决的问题。

## 查询路由与自适应检索（Query Routing & Adaptive Retrieval）

查询路由解决"走哪条检索链路"，自适应检索解决"走多深"。两者都是检索前的策略决策，放在一起形成完整的"检索前决策层"。

### 规则路由
按关键词把问题分发到不同索引/工具，简单稳定。

### LLM 路由
先让 LLM 判断“该走哪条检索链路”，再执行对应分支，灵活但更依赖模型判断。

### 边界
- 适合：多知识源、多工具场景
- 局限：错误路由会直接影响最终答案

### 关键决策点
当前问题应走哪条索引/工具链路，才能用最少噪声拿到证据。


In [ ]:
def routing_pipeline(question: str) -> str:
    """教学版路由：LLM 选择 math_index / general_index 分支；本节两条分支共用同一 retriever。"""
    trace: list[Step] = []

    router_prompt = (
        "请判断下列问题应该路由到哪个索引：\n"
        "- math_index: 数学公式、推导过程相关\n"
        "- general_index: 通用概念、应用背景相关\n"
        f"问题：{question}\n"
        "仅输出索引名称。"
    )
    target = llm_call(router_prompt).strip().lower()
    branch = "math_index" if "math" in target else "general_index"
    trace.append(Step("route", {"target": branch}))

    docs = retriever.invoke(question)
    ctx = trim_context_to_budget("\n\n".join(d.page_content for d in docs), CONTEXT_CHAR_BUDGET)
    trace.append(Step("retrieve", {"branch": branch, "n_hits": len(docs)}))

    ans = llm_call(build_rag_generation_prompt(question, ctx))
    trace.append(Step("finalize", {"reason": "answered"}))
    routing_pipeline.last_trace = trace
    return ans


def inspect_routing(question: str) -> None:
    print(f"❓ 问题: {question}\n")
    ans = routing_pipeline(question)
    print(f"🧠 最终答案：\n{ans}\n")
    print("🔍 流程 trace：")
    print_trace(routing_pipeline.last_trace)


### 查询路由结果分析

查询路由的价值在于：当系统有多个专用索引时，先选对索引再检索，比把所有内容混在一起检索更精准。规则路由简单稳定但不够灵活，LLM 路由更通用但引入了模型判断的不确定性。

查询路由决定了"走哪条路"，但还有一个问题没解决：**走多深？** 一个简单的事实查询只需要一次检索就够了，但一个复杂的多维度问题可能需要迭代或递归检索。让系统根据问题复杂度自动选择检索深度，就是自适应检索要做的事情。

## 自适应检索（Adaptive Retrieval）

### 失败场景
对所有问题都用同一个检索深度——简单问题浪费资源，复杂问题覆盖不足。

### 思路
先让 LLM 判断问题的复杂度（simple / moderate / complex），再根据复杂度自动选择检索策略：简单问题单次检索，中等问题迭代检索，复杂问题递归检索。

### 与查询路由的关系
查询路由选择"走哪条路"（哪个索引/工具），自适应检索选择"走多深"（单次/迭代/递归）。两者组合形成完整的"检索前决策层"。

### 边界
- 适合：问题复杂度差异大的场景
- 局限：复杂度判断依赖 LLM，可能误判

### 关键决策点
当前问题需要多深的检索策略才能覆盖答案所需的全部证据。

In [ ]:
def adaptive_pipeline(question: str) -> str:
    """LLM 先给 simple/moderate/complex 标签，再调度到单次/迭代/递归 pipeline。"""
    trace: list[Step] = []

    classifier_prompt = (
        "请判断下列问题的复杂度，仅输出: simple / moderate / complex\n"
        "- simple: 事实查询，单一知识点\n"
        "- moderate: 需要对比或补齐信息\n"
        "- complex: 多维度复杂问题，需拆解\n"
        f"问题：{question}"
    )
    raw = llm_call(classifier_prompt).strip().lower()
    if "complex" in raw:
        chosen = "complex"
    elif "moderate" in raw:
        chosen = "moderate"
    else:
        chosen = "simple"
    trace.append(Step("classify", {"complexity": chosen}))

    if chosen == "complex":
        ans = recursive_pipeline(question)
        trace.append(Step("delegate", {"to": "recursive_pipeline"}))
    elif chosen == "moderate":
        ans = iterative_pipeline(question)
        trace.append(Step("delegate", {"to": "iterative_pipeline"}))
    else:
        docs = retriever.invoke(question)
        ctx = trim_context_to_budget("\n\n".join(d.page_content for d in docs), CONTEXT_CHAR_BUDGET)
        ans = llm_call(build_rag_generation_prompt(question, ctx))
        trace.append(Step("delegate", {"to": "single_pass"}))

    trace.append(Step("finalize", {"reason": f"via_{chosen}"}))
    adaptive_pipeline.last_trace = trace
    return ans


def inspect_adaptive(question: str) -> None:
    print(f"❓ 问题: {question}\n")
    ans = adaptive_pipeline(question)
    print(f"🧠 最终答案：\n{ans}\n")
    print("🔍 流程 trace：")
    print_trace(adaptive_pipeline.last_trace)


### 自适应检索结果分析

可以看到，简单问题被正确判定为 simple 并用单次检索高效处理，复杂问题则自动升级为递归检索以覆盖更多维度。自适应检索本质上是一个"调度器"，它复用了前面介绍的迭代和递归检索作为底层策略。

到目前为止，我们解决了"多走几步"和"走对方向"的问题。但还有一个更根本的问题没有触及：**检索回来的结果质量够不够好？** 如果检索结果里混入了大量低相关片段，无论后续流程多精巧，最终答案都可能被噪声带偏。Corrective RAG 就是在生成前加一道质量把关。

## Corrective RAG

### 失败场景
检索结果里混入低相关片段，导致答案偏题或不稳定。

### CRAG 流程
1. 先检索候选文档
2. 用 grader 评估每条证据相关性
3. 按 `全部相关 / 部分相关 / 无关` 分支处理
4. 再生成最终回答

### 直接收益
在不大改系统结构的前提下，先把“坏上下文”挡在生成前。

### 关键决策点
候选检索结果是全部可用、部分可用还是基本不可用。


In [ ]:
def _grade_relevance(question: str, text: str) -> str:
    judge_prompt = (
        "判断下面文档片段和问题的相关性，只输出：relevant / partial / irrelevant\n"
        f"问题：{question}\n片段：{text[:600]}"
    )
    raw = llm_call(judge_prompt).lower()
    if "irrelevant" in raw:
        return "irrelevant"
    if "partial" in raw:
        return "partial"
    return "relevant"


def crag_pipeline(question: str) -> str:
    trace: list[Step] = []
    docs = retriever.invoke(question)
    bucket = {"relevant": [], "partial": [], "irrelevant": []}

    for i, d in enumerate(docs):
        tag = _grade_relevance(question, d.page_content)
        bucket[tag].append(d.page_content)
        trace.append(Step("grade", {"idx": i + 1, "tag": tag}))

    if len(bucket["relevant"]) == len(docs):
        pieces = bucket["relevant"]
        branch = "all_relevant"
    elif bucket["relevant"] or bucket["partial"]:
        pieces = (bucket["relevant"] + bucket["partial"])[:4]
        branch = "partially_relevant"
    else:
        pieces = ["检索证据不足，请先重写问题后再检索。"]
        branch = "irrelevant"
    trace.append(Step("route", {"branch": branch, "n_used": len(pieces)}))

    ctx = trim_context_to_budget("\n\n".join(pieces), CONTEXT_CHAR_BUDGET)
    ans = llm_call(build_rag_generation_prompt(question, ctx))
    trace.append(Step("finalize", {"reason": branch}))
    crag_pipeline.last_trace = trace
    return ans


def inspect_crag(question: str) -> None:
    print(f"❓ 问题: {question}\n")
    ans = crag_pipeline(question)
    print(f"🧠 最终答案：\n{ans}\n")
    print("🔍 流程 trace：")
    print_trace(crag_pipeline.last_trace)


### Corrective RAG 结果分析

CRAG 的三种分支各有含义：
- **all_relevant**：所有检索结果都相关，直接用于生成——这是最理想的情况。
- **partially_relevant**：部分相关、部分噪声，过滤掉低质量片段后再生成——这是最常见的情况。
- **irrelevant**：检索结果基本不可用，应提示用户重写问题或触发补充检索。

CRAG 的决策是单次的：评估一次、处理一次。但如果我们希望系统能**持续自我判断**——"这个回答够好吗？要不要再检索一轮？"——就需要一个更动态的反思机制。这就是 Self-RAG 的核心思想。

## Self-RAG（教学化简版）

Self-RAG 的核心是三段循环：**retrieve -> generate -> critique**。

- 训练阶段：通过特定监督信号学习何时继续检索、何时停止。
- 推理阶段：根据当前回答质量 decide 是否追加检索。

下图用于理解原始论文思路（保留原图链接）：
- `./figures/selfrag.png`
- `./figures/selftoken.jpg`

本节不再要求本地 LLaMA2 / GGUF / 重型 pack 环境，只保留可运行控制逻辑示例。

### 关键决策点
当前回答质量是否足够，是否值得继续追加检索。


In [ ]:
def self_rag_pipeline(question: str, max_steps: int = 2) -> str:
    """教学化简版 Self-RAG：retrieve -> generate-and-critique 循环，靠 [FINISH]/[CONTINUE] 标记决策。"""
    trace: list[Step] = []
    context_list: list[str] = []
    current_answer = ""

    for i in range(max_steps):
        search_q = question if not current_answer else f"{question} (补充检索以完善: {current_answer[:50]}...)"
        docs = retriever.invoke(search_q)
        context_list.extend(d.page_content for d in docs)
        ctx = trim_context_to_budget("\n\n".join(context_list[-4:]), CONTEXT_CHAR_BUDGET)
        trace.append(Step("retrieve", {"step": i + 1, "n_hits": len(docs)}))

        prompt = (
            f"问题：{question}\n上下文：{ctx}\n当前草稿：{current_answer}\n\n"
            "任务：\n"
            "1. 如果当前上下文足以完整、准确地回答问题，请给出最终答案并以 [FINISH] 结尾。\n"
            "2. 如果不足，请给出当前已知的片段，并以 [CONTINUE] 结尾，指出还需检索什么。"
        )
        response = llm_call(prompt)
        current_answer = response

        is_finish = "[FINISH]" in response
        trace.append(Step("reflect", {"step": i + 1, "verdict": "FINISH" if is_finish else "CONTINUE"}))
        if is_finish:
            trace.append(Step("finalize", {"reason": "self_finish"}))
            self_rag_pipeline.last_trace = trace
            return response.replace("[FINISH]", "").strip()

    trace.append(Step("finalize", {"reason": "max_steps_reached"}))
    self_rag_pipeline.last_trace = trace
    return current_answer.replace("[CONTINUE]", "").strip()


def inspect_self_rag(question: str) -> None:
    print(f"❓ 问题: {question}\n")
    ans = self_rag_pipeline(question)
    print(f"🧠 最终答案：\n{ans}\n")
    print("🔍 流程 trace：")
    print_trace(self_rag_pipeline.last_trace)


### Self-RAG 结果分析

Self-RAG 的优势在于：模型自己决定何时停止检索，避免了固定轮次带来的浪费或不足。

**教学简化版 vs 论文原版的差异：** 本节用通用 LLM API 模拟了 Self-RAG 的 retrieve → generate → critique 循环。论文原版通过特殊的 reflection tokens（如 `[Retrieve]`、`[IsRel]`、`[IsSup]`）在模型内部完成这些决策，需要专门微调的模型（如 selfrag_llama2_7b）。教学版保留了核心控制逻辑，但决策精度依赖通用 LLM 的判断能力，而非内置的反思 token。

## 小结：六种流程增强如何选择

- `迭代检索`：适合“先答后补”的渐进式问题。
- `递归检索`：适合可拆解的复杂问题。
- `查询路由`：适合多索引/多工具环境。
- `Corrective RAG`：适合先提升检索质量再生成。
- `Self-RAG`：适合需要模型自我反思控制检索轮次的场景。
- `自适应检索`：适合需要根据问题复杂度自动选择检索深度的场景，与查询路由配合使用。

## 学习检查点

- 你能描述 CRAG 的三种分支（all / partial / irrelevant）吗？
- 你能解释 Self-RAG 在“继续检索/停止检索”上的决策点吗？
- 你能解释查询路由（选方向）和自适应检索（选深度）的区别吗？
- 你知道何时应从流程增强升级到系统增强吗？

## 全方法对比表

| 方法 | 控制类型 | 典型修复问题 | 新增复杂度 | 最适合场景 |
|---|---|---|---|---|
| 迭代检索 | 补检索型 | 首轮回答不完整 | 中 | 信息可逐步补齐 |
| 递归检索 | 拆任务型 | 复杂问题可分解 | 中-高 | 多跳问答 |
| 查询路由 | 选路径型 | 多数据源或多索引 | 中 | 中大型系统 |
| 自适应检索 | 选深度型 | 问题复杂度差异大 | 中 | 与查询路由配合 |
| Corrective RAG | 控质量型 | 检索结果噪声多 | 中-高 | 质量优先场景 |
| Self-RAG | 自反思型 | 需要动态控制检索 | 高 | 高要求场景 |

In [ ]:
iterative_df = eval_pipeline(iterative_pipeline, qna_dict)
recursive_df = eval_pipeline(recursive_pipeline, qna_dict)
routing_df   = eval_pipeline(routing_pipeline,   qna_dict)
crag_df      = eval_pipeline(crag_pipeline,      qna_dict)
selfrag_df   = eval_pipeline(self_rag_pipeline,  qna_dict)
adaptive_df  = eval_pipeline(adaptive_pipeline,  qna_dict)


In [ ]:
compare_df = build_compare_table(
    [iterative_df, recursive_df, routing_df, crag_df, selfrag_df, adaptive_df],
    names=["iterative", "recursive", "routing", "crag", "self_rag", "adaptive"],
)
compare_df


## 跨方法对比读法

`compare_df` 每行一题、每列一个方法的 0~2 分（裁判使用维度计分 prompt）。
注意：本节 6 个方法各有自己的"擅长场景"——比如递归适合可显式拆解的问题，
路由适合多源场景。同题分数高低不是绝对优劣指标，应结合 inspect 输出的 trace
理解"流程是否走对"。


## 与系统增强的衔接

当你发现“仅靠流程决策还不够”，例如需要跨多个文档智能体协作、需要长期会话记忆时，请继续学习：`3. 系统增强.ipynb`。